# boundaries

> Wall-clock-aware segment boundary computation: group VAD speech chunks into segments cut at silence-gap midpoints.

Pure logic — no capability calls. This is the **final home** of the algorithm originally validated in `cjm-transcription-audio-segment`'s `AudioSegmentService.compute_segment_boundaries` (that copy lives in a FastHTML-adjacent package and is slated to be consumed from here during the consumer cascade — see the pass-2 evidence ledger).

In [ ]:
#| default_exp boundaries

In [ ]:
#| export
from typing import Dict, List, Optional

In [ ]:
#| export
def compute_segment_boundaries(
    vad_chunks: List[Dict[str, float]],  # [{start, end, ...}, ...] sorted by start
    max_segment_duration: float,         # Target max wall-clock segment length in seconds
    audio_duration: float,               # Full audio duration in seconds
) -> List[Dict[str, float]]:  # [{start, end}, ...] covering [0, audio_duration]
    """Group VAD chunks into segments cut at silence-gap midpoints.

    **Wall-clock-aware, pre-emptive cuts.** `max_segment_duration` caps the
    wall-clock duration of each output segment (not the speech-only duration
    within it) — matching the downstream forced-alignment constraint, which
    operates on the resulting audio file's length.

    Algorithm:
      1. If audio_duration <= max_segment_duration or no chunks: single segment
         covering [0, audio_duration].
      2. Walk chunks sequentially. For each chunk, check whether accepting it
         would push the in-progress segment's wall-clock duration over max. If
         so AND we already have content: cut **before** this chunk at the
         silence-gap midpoint between the previous chunk's end and this chunk's
         start (chunks that abut with no gap cut exactly at the previous
         chunk's end).
      3. The final segment extends to audio_duration.

    **Wall-clock invariant.** Every NON-FINAL segment's wall-clock duration is
    <= max_segment_duration. The final segment may exceed max only because it
    extends to audio_duration to cover trailing silence. A single VAD chunk
    whose own duration exceeds max forms a segment of its native length —
    speech is never split mid-chunk.
    """
    if audio_duration <= 0:
        return []
    if audio_duration <= max_segment_duration or not vad_chunks:
        return [{"start": 0.0, "end": float(audio_duration)}]

    boundaries: List[Dict[str, float]] = []
    segment_start = 0.0
    prev_chunk_end: Optional[float] = None

    for chunk in vad_chunks:
        chunk_start = float(chunk["start"])
        chunk_end = float(chunk["end"])

        # Pre-emptive cut: if accepting this chunk would push the segment's
        # wall-clock duration over max AND we have prior content, cut now.
        if prev_chunk_end is not None and (chunk_end - segment_start) > max_segment_duration:
            cut_point = (
                (prev_chunk_end + chunk_start) / 2.0
                if chunk_start > prev_chunk_end
                else prev_chunk_end
            )
            boundaries.append({"start": segment_start, "end": cut_point})
            segment_start = cut_point

        prev_chunk_end = chunk_end

    boundaries.append({"start": segment_start, "end": float(audio_duration)})
    return boundaries

In [ ]:
# Behavioral checks (mirror the validated audio-segment semantics)

# Short audio -> single covering segment
assert compute_segment_boundaries([{"start": 1.0, "end": 5.0}], 300.0, 28.0) == [
    {"start": 0.0, "end": 28.0}
]

# No chunks -> single covering segment
assert compute_segment_boundaries([], 300.0, 28.0) == [{"start": 0.0, "end": 28.0}]

# Zero duration -> no segments
assert compute_segment_boundaries([], 300.0, 0.0) == []

# Two chunks straddling the cap -> cut at the silence-gap midpoint
chunks = [{"start": 0.0, "end": 100.0}, {"start": 120.0, "end": 200.0}]
b = compute_segment_boundaries(chunks, 150.0, 200.0)
assert b == [{"start": 0.0, "end": 110.0}, {"start": 110.0, "end": 200.0}], b

# Non-final wall-clock invariant holds on a longer synthetic chunk train
chunks = [{"start": float(i * 10), "end": float(i * 10 + 8)} for i in range(60)]
b = compute_segment_boundaries(chunks, 60.0, 600.0)
assert all((s["end"] - s["start"]) <= 60.0 for s in b[:-1]), b
assert b[0]["start"] == 0.0 and b[-1]["end"] == 600.0
len(b)

10